In [1]:
import cv2
import os

# Folder containing your .png images
IMG_DIR = "Annotated"
OUT_DIR = "Annotated/labels"
os.makedirs(OUT_DIR, exist_ok=True)

scale = .25
display_img = None
orig_img = None
drawing = False
ix, iy = -1, -1
boxes = []

def draw_box(event, x, y, flags, param):
    global ix, iy, drawing, boxes, display_img
    if event == cv2.EVENT_LBUTTONDOWN:
        drawing = True
        ix, iy = x, y
    elif event == cv2.EVENT_LBUTTONUP:
        drawing = False
        x_min, y_min = min(ix, x), min(iy, y)
        x_max, y_max = max(ix, x), max(iy, y)
        
        x_min_orig = int(x_min / scale)
        y_min_orig = int(y_min / scale)
        x_max_orig = int(x_max / scale)
        y_max_orig = int(y_max / scale)

        boxes.append((x_min_orig, y_min_orig, x_max_orig, y_max_orig))

        # Draw rectangle on display image
        cv2.rectangle(display_img, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)
        cv2.imshow("image", display_img)


def save_yolo_format(img_name, boxes, img_w, img_h):
    label_path = os.path.join(OUT_DIR, os.path.splitext(img_name)[0] + ".txt")
    with open(label_path, "w") as f:
        for (x1, y1, x2, y2) in boxes:
            x_center = ((x1 + x2) / 2) / img_w
            y_center = ((y1 + y2) / 2) / img_h
            w = (x2 - x1) / img_w
            h = (y2 - y1) / img_h
            f.write(f"0 {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}\n")

for img_name in os.listdir(IMG_DIR):
    if not img_name.endswith(".jpg"):
        continue
    img_path = os.path.join(IMG_DIR, img_name)
    orig_img = cv2.imread(img_path)
    display_img = cv2.resize(orig_img, (int(orig_img.shape[1] * scale),
                                       int(orig_img.shape[0] * scale)))
    clone = display_img.copy()
    boxes = []

    cv2.namedWindow("image")
    cv2.setMouseCallback("image", draw_box)

    while True:
        cv2.imshow("image", display_img)
        key = cv2.waitKey(1) & 0xFF
        if key == ord("s"):  # save and move on
            save_yolo_format(img_name, boxes, orig_img.shape[1], orig_img.shape[0])
            break
        elif key == ord("r"):  # reset boxes
            display_img = clone.copy()
            boxes = []
            print("reset boxes")
        elif key == 27:  # ESC to quit
            cv2.destroyAllWindows()
            exit()

cv2.destroyAllWindows()

reset boxes
reset boxes
